# 5 Gate, highway, or route: one idea from three papers

The gate/highway/MoE family composes with any `(batch, hidden)` tensor — images, text, tabular data, anything. It sits here because
this lesson already has a window and a baseline table to beat, the same reason
`1_pytorch_intro/04_search` introduced ray.tune where a two-second training run made it
obviously worth the investment rather than because tuning is somehow a time-series idea.


## 5.1 One idea, three resolutions

| component | mechanism | where it comes from |
|---|---|---|
| `Gate` | `sigmoid(Wx) * x` -- a learned per-dimension scale | the gate inside every LSTM cell (Hochreiter & Schmidhuber, 1997) and the reset/update gates of the GRU (Cho et al., 2014, *Learning Phrase Representations using RNN Encoder-Decoder for Statistical Machine Translation*) |
| `Highway` | `g * T(x) + (1 - g) * x` -- a learned interpolation between a transform and the identity | Highway Networks (Srivastava, Greff & Schmidhuber, 2015), built explicitly to let very deep plain networks train at all -- the direct ancestor of the ResNet residual connection, which is a Highway with the gate fixed at 0.5 |
| `MoE` | `sum_i softmax(Wx)_i * expert_i(x)` -- N gates routing between N experts | Jacobs & Jordan, 1991, *Adaptive Mixtures of Local Experts* -- the **dense** version tested in §5.2-5.4 below: all N experts run, every time. Modern **sparse** MoE (Shazeer et al., 2017, *Outrageously Large Neural Networks*; the Switch Transformer, Fedus et al., 2021) routes each token to only `k` of thousands of experts -- capacity scales with N, compute scales with k. §5.5 builds and measures that version (`vectormesh`’s `SparseMoE`), which is the part that actually makes production MoE fast at scale |

Read top to bottom, each row is the row above with more capacity to choose from: one
gate choosing between a transform and nothing (`Gate`), one gate choosing between a
transform and the identity (`Highway`), N gates choosing between N transforms (`MoE`).

This lesson's electricity-demand series has real daily and weekly structure -- exactly
the kind of thing a router could plausibly use. Predict before you look at the table:
does giving the mechanism something real to route on change the verdict?


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from vectormesh.components import Gate, Highway, MoE, NeuralNet, SparseMoE
from vectormesh.components.metrics import MAE
from vectormesh.training import Trainer, TrainerSettings

from australian_electricity import load_ausalia
from windowing import window_index
from notebooktester import param

DAY = 48  # half-hourly steps per day
W, H = 4 * DAY, DAY  # four days of history, one day of horizon -- same window
# 6_uncertainty/01_probabilistic uses, so results sit next to that notebook's
HIDDEN = 64
STRIDE = 24
SEEDS = (0, 1, 2, 3, 4)

ausalia = load_ausalia()
mu = ausalia.values[ausalia.splits["train"]].mean()
sigma = ausalia.values[ausalia.splits["train"]].std()


def windows(split: str, stride: int):
    values = (ausalia.values[ausalia.splits[split]] - mu) / sigma
    win = values[window_index(len(values), W + H, stride)]
    return win[:, :W], win[:, W:]


xtr, ytr = windows("train", STRIDE)
xva, yva = windows("val", STRIDE * 2)
print(f"train {tuple(xtr.shape)}   val {tuple(xva.shape)}")

# one batch covering the whole val split -- every Trainer below shares this loader
validloader = DataLoader(TensorDataset(xva, yva), batch_size=len(xva))


def mae(y: torch.Tensor, yhat: torch.Tensor) -> float:
    return float((y - yhat).abs().mean())


def make_model(
    mechanism: str, hidden: int = HIDDEN, n_experts: int = 4, k: int = 1
) -> nn.Module:
    if mechanism == "plain":
        return NeuralNet(hidden_size=[W, hidden], out_size=H)
    if mechanism == "deeper":
        # capacity-matched control: one more Linear -> GELU than "plain", no gate at all.
        # Gate's own weight matrix costs exactly what this second layer costs at this width.
        return NeuralNet(hidden_size=[W, hidden, hidden], out_size=H)
    inp, act, out = nn.Linear(W, hidden), nn.GELU(), nn.Linear(hidden, H)
    if mechanism == "gate":
        mid = Gate(hidden_size=hidden)
    elif mechanism == "highway":
        mid = Highway(transform=NeuralNet(hidden, hidden), hidden_size=hidden)
    elif mechanism == "moe":
        mid = MoE(
            experts=[nn.Linear(hidden, hidden) for _ in range(n_experts)],
            hidden_size=hidden,
            out_size=hidden,
        )
    elif mechanism == "sparse_moe":
        mid = SparseMoE(
            experts=[nn.Linear(hidden, hidden) for _ in range(n_experts)],
            hidden_size=hidden,
            out_size=hidden,
            k=k,
        )
    else:
        raise ValueError(mechanism)
    return nn.Sequential(inp, act, mid, out)


def run(
    mechanism: str,
    seed: int,
    epochs: int,
    lr: float = 1e-3,
    batch_size: int = 64,
    **model_kwargs,
) -> float:
    torch.manual_seed(seed)
    model = make_model(mechanism, **model_kwargs)
    trainloader = DataLoader(
        TensorDataset(xtr, ytr), batch_size=batch_size, shuffle=True
    )
    settings = TrainerSettings(
        epochs=epochs,
        metrics=[MAE()],
        logdir=Path("modellogs").resolve(),
        train_steps=len(trainloader),
        valid_steps=len(validloader),
        optimizer_kwargs={"lr": lr},
        earlystop_kwargs=None,
    )
    trainer = Trainer(
        model=model,
        settings=settings,
        loss_fn=nn.MSELoss(),
        optimizer=torch.optim.Adam,
        traindataloader=trainloader,
        validdataloader=validloader,
        progress=False,
    )
    return trainer.loop().metric_dict["MAE"]


def mae_curve(
    mechanism: str,
    epochs: int,
    seed: int = 0,
    lr: float = 1e-3,
    batch_size: int = 64,
    **model_kwargs,
) -> tuple[list[float], list[float]]:
    """Same training as `run`, but returns train and val MAE after *every* epoch instead
    of only the last one -- the curve a single final number can't show. A Reporter
    captures both: metric_dict already carries val MAE per epoch, and the closure does one
    extra forward pass over train for the train-side curve."""
    torch.manual_seed(seed)
    model = make_model(mechanism, **model_kwargs)
    trainloader = DataLoader(
        TensorDataset(xtr, ytr), batch_size=batch_size, shuffle=True
    )
    train_curve: list[float] = []
    val_curve: list[float] = []

    def track(epoch, train_loss, test_loss, metric_dict):
        val_curve.append(metric_dict["MAE"])
        with torch.no_grad():
            train_curve.append(mae(ytr, model(xtr)))

    settings = TrainerSettings(
        epochs=epochs,
        metrics=[MAE()],
        logdir=Path("modellogs").resolve(),
        train_steps=len(trainloader),
        valid_steps=len(validloader),
        optimizer_kwargs={"lr": lr},
        earlystop_kwargs=None,
    )
    trainer = Trainer(
        model=model,
        settings=settings,
        loss_fn=nn.MSELoss(),
        optimizer=torch.optim.Adam,
        traindataloader=trainloader,
        validdataloader=validloader,
        reporters=[track],
        progress=False,
    )
    trainer.loop()
    return train_curve, val_curve

## 5.2 How many epochs is enough? Look at the curve, then check it across seeds

The normal way to answer this -- the way you would for any model -- is to look at
the train and val MAE across epochs: is it still improving, has it plateaued, is
train pulling away from val (overfitting)? Do that first, below, for `gate` and its
capacity-matched control `deeper`.

In [ ]:
CURVE_EPOCHS = param(90, test=8)

fig, ax = plt.subplots(figsize=(9, 3))
colors = {"deeper": "C0", "gate": "C1"}
for name in ("deeper", "gate"):
    train_curve, val_curve = mae_curve(name, epochs=CURVE_EPOCHS)
    epochs_axis = range(1, len(val_curve) + 1)
    ax.plot(epochs_axis, train_curve, color=colors[name], lw=0.8, ls=":", alpha=0.6)
    ax.plot(epochs_axis, val_curve, color=colors[name], lw=1.4, label=name)
ax.set_xlabel("epoch")
ax.set_ylabel("MAE  (dotted = train, solid = val)")
ax.legend(fontsize=8)
plt.tight_layout()

Both curves drop steeply for about the first 20 epochs (val MAE ~0.177 for both
models by epoch 20) then flatten into a noisy plateau through epoch 90 (val MAE
~0.150 for both at epoch 90, but bouncing epoch-to-epoch by several hundredths --
more than the eventual gap between the two models). Train sits consistently below
val by a modest, stable gap (~0.025 by epoch 90 for both) that does not visibly
widen after the initial drop -- a mild, stable generalisation gap, not runaway
overfitting.

That noisy plateau is itself the visual argument for the seed check below: reading
either curve at a single epoch could land on any of several nearby values just from
which epoch you happened to pick, for either model.

In [ ]:
EPOCHS_TO_CHECK = param((30, 60, 90), test=(1, 2))

for epochs in EPOCHS_TO_CHECK:
    for name in ("gate", "deeper"):
        scores = [run(name, seed, epochs) for seed in SEEDS]
        print(
            f"epochs={epochs:3d}   {name:7s} {np.mean(scores):.5f} +/- {np.std(scores):.5f}"
        )
    print()

`gate` and `deeper` -- its capacity-matched control, same parameter count, no gate
anywhere -- land inside each other's seed spread at every epoch count checked here.
That is the answer: pick a reasonably converged epoch count and stop looking for a
number where the gap opens up, because the checks above say there is not a stable one
to find. `EPOCHS = 90` below is not the best number in any of these rows; it is simply
past the region where training is still visibly moving.

`highway`, dense `moe`, and both `sparse_moe` configurations below are trained at this
same settled `EPOCHS=90` without an individual convergence check each -- consistent
with how `highway` and `moe` were already handled here before this section existed,
not a new shortcut introduced only for the newest architecture.

## 5.3 The full family, at a settled epoch count

In [ ]:
EPOCHS = param(90, test=2)

# (label, mechanism, make_model/run kwargs) -- sparse_moe appears twice: once at the
# same N as dense moe (does routing beat blending at matched capacity?), once at 16x
# the experts (does the extra capacity a router can afford almost for free help?).
MODEL_CONFIGS = [
    ("plain", "plain", {}),
    ("deeper", "deeper", {}),
    ("gate", "gate", {}),
    ("highway", "highway", {}),
    ("moe", "moe", {"n_experts": 4}),
    ("sparse_moe (N=4)", "sparse_moe", {"n_experts": 4, "k": 1}),
    ("sparse_moe (N=64)", "sparse_moe", {"n_experts": 64, "k": 1}),
]

results: dict[str, dict] = {}
for label, mechanism, kwargs in MODEL_CONFIGS:
    scores = [run(mechanism, seed, EPOCHS, **kwargs) for seed in SEEDS]
    params = sum(p.numel() for p in make_model(mechanism, **kwargs).parameters())
    results[label] = {
        "mae": float(np.mean(scores)),
        "std": float(np.std(scores)),
        "params": params,
    }
    print(
        f"{label:20s} {results[label]['mae']:.5f} +/- {results[label]['std']:.5f}   {params:>8,} params"
    )

## 5.4 The table

In [ ]:
order = sorted(results, key=lambda k: results[k]["mae"])
print(f"{'model':20s} {'val MAE':>9} {'+/-':>7} {'params':>9}")
print("-" * 50)
for name in order:
    r = results[name]
    print(f"{name:20s} {r['mae']:9.5f} {r['std']:7.5f} {r['params']:9,}")

`deeper` and `gate` are still the two best rows, sitting inside each other's seed
spread -- exactly the §5.2 finding, now with the rest of the family for context.
`highway` costs 1.8x `plain`'s parameters and scores worse; dense `moe` costs 2.1x
and scores worse still.

**Sparse routing does not rescue the family here -- if anything it's the weakest of
the trained mechanisms.** `sparse_moe (N=4)` -- the *same* parameter count as dense
`moe`, just routed instead of blended -- scores 0.1696 against dense `moe`'s 0.1647,
a difference well inside both models' seed noise (std 0.0113 and 0.0101 respectively):
routing does not beat blending at matched capacity on this task. Growing capacity 16x
further, `sparse_moe (N=64)` at 285,872 params (18.5x `plain`, 8.8x dense `moe`)
scores 0.1704 -- statistically indistinguishable from the 4-expert version, and still
the worst trained model apart from `plain` itself. Sixteen times the experts bought
nothing.

Checked directly why, for one seed (seed=0, 90 epochs -- the same seed §5.2's curve
used): on the 361 validation windows, the trained `sparse_moe (N=64)` router only
ever chose **15 of its 64 experts** -- the top expert alone took 153 of 361 examples
(42%), the top five together took 324 (90%). This is router collapse, a known real
failure mode of top-1 routing: nothing in `SparseMoE` (or this notebook's training
loop) penalises an unbalanced router, so once a handful of experts start winning
early they keep winning, and the other 49 experts train on close to nothing.
Production Switch/Mixtral implementations add an auxiliary load-balancing loss for
exactly this reason; this notebook's `SparseMoE` doesn't, and the 64-expert row above
is the visible cost of that gap (see E6).

A series with real daily and weekly structure -- exactly the kind of thing a router
could plausibly use -- still does not make any member of this family (dense or
sparse) earn its parameters, once checked properly. The same shape of result holds
on a four-patch chunked window: routing is a real
bet that structure exists *and that this architecture can find and use it*, and the
bet does not pay just because the structure is there or the mechanism is more
sophisticated.

## 5.5 Is sparse routing cheap to scale?

§5.4 just gave the concrete answer to "does it forecast better": no, not at `k=1`
on this task, and part of why not was router collapse rather than the routing idea
itself. This section asks a separate, still real question: even given that verdict,
does the mechanism deliver on its actual production promise -- holding far more
capacity than any single input pays for in compute? `MoE` is *dense*: every expert
runs on every input, so adding experts always costs more compute -- by construction,
it cannot show that decoupling. `vectormesh`'s `SparseMoE` (0.7.0) routes each example
to its top-`k` experts with real gather/scatter -- an expert with zero examples routed
to it this batch does zero work, not masked-out work -- so compute per example scales
with `k`, not with the number of experts `N`.

The test: fix `k=1` (Switch-style) and a fixed batch of this notebook's own windows,
then grow `N` from 4 to 256 experts. If the sparse claim is real, wall-clock time per
forward pass should stay roughly flat while parameter count grows 64x. Dense `MoE`,
run on the identical batch and the identical `N` values, is the control here -- it has
no reason to stay flat, since it always runs every expert regardless of how many
exist.

In [ ]:
import time

from vectormesh.components import SparseMoE

N_EXPERTS_TO_CHECK = param((4, 16, 64, 256), test=(2, 4))
ROUTING_BATCH = 64
REPS = param(30, test=3)

# a real batch of this notebook's own windows, projected to HIDDEN once -- the
# component under test is the mixture-of-experts block, not the input projection
probe_input = nn.Linear(W, HIDDEN)
probe_x = probe_input(xtr[:ROUTING_BATCH])


def time_forward(model: nn.Module, x: torch.Tensor, reps: int) -> float:
    for _ in range(5):
        model(x)
    start = time.perf_counter()
    for _ in range(reps):
        model(x)
    return (time.perf_counter() - start) / reps * 1000


sparse_timings: dict[int, dict] = {}
dense_timings: dict[int, dict] = {}
for n_experts in N_EXPERTS_TO_CHECK:
    torch.manual_seed(0)
    sparse = SparseMoE(
        experts=[nn.Linear(HIDDEN, HIDDEN) for _ in range(n_experts)],
        hidden_size=HIDDEN,
        out_size=HIDDEN,
        k=1,
    )
    torch.manual_seed(0)
    dense = MoE(
        experts=[nn.Linear(HIDDEN, HIDDEN) for _ in range(n_experts)],
        hidden_size=HIDDEN,
        out_size=HIDDEN,
    )

    sparse_ms = time_forward(sparse, probe_x, REPS)
    dense_ms = time_forward(dense, probe_x, REPS)
    params = sum(p.numel() for p in sparse.parameters())  # identical count, both sides

    sparse_timings[n_experts] = {"ms": sparse_ms, "params": params}
    dense_timings[n_experts] = {"ms": dense_ms, "params": params}
    print(
        f"N={n_experts:4d} experts   sparse {sparse_ms:7.3f} ms   "
        f"dense {dense_ms:7.3f} ms   {params:>9,} params"
    )

Measured on this machine (batch of 64 of this notebook's own windows, hidden=64,
k=1, 30 reps after 5 warmup calls):

| N experts | sparse ms/fwd | dense ms/fwd | params |
|---|---|---|---|
| 4 | 0.135 | 0.104 | 16,900 |
| 16 | 0.222 | 0.213 | 67,600 |
| 64 | 0.171 | 0.520 | 270,400 |
| 256 | 0.258 | 1.778 | 1,081,600 |

From N=4 to N=256, parameter count grows **64x**. `SparseMoE`'s forward time grows
only **1.9x** (0.135 -> 0.258 ms) -- routing overhead (one `argsort` plus a
per-active-expert Python loop), not added compute, since each example still passes
through exactly one expert. Dense `MoE`, run on the identical batch and the identical
`N` values, grows **17.1x** (0.104 -> 1.778 ms) -- close to proportional with `N`, as
it must be, since it evaluates every expert on every example regardless of capacity.
Dense is faster at N=4 and N=16 (routing machinery costs more than it saves when there
is little to route around); sparse pulls ahead by N=64 (3.0x faster) and is **6.9x**
faster than dense by N=256, for identical parameter counts.

So the two questions this notebook now asks about MoE get two different answers.
"Does it forecast better here" -- §5.4 -- is no, and router collapse explains part of
why. "Is the compute/capacity decoupling real" -- this section -- is yes, measured
directly: `SparseMoE` holds 64x the parameters of its smallest configuration for well
under 2x the compute, which is the actual mechanism production MoE relies on, however
this specific forecasting task's verdict on it came out.

`04_search` is where you stop checking one hand-picked configuration at a time and let
a search cover the space properly.

## What to take from this

- Gate, Highway and MoE are one idea -- a learned keep-or-replace decision -- at
  increasing resolution, and each one traces to a specific paper rather than floating
  free in a library's component list.
- The mechanism is general-purpose. It showed up here because this lesson had a window
  and a baseline ready, not because time series needs it more than any other modality.
- **Look at the curve before you trust a number.** §5.2 opens with the train/val MAE
  curve any human would reach for first, then checks that the visual impression holds
  across seeds -- because this notebook's own history says a convincing-looking single
  check can still be wrong.
- Whether gating (or routing) earns its extra parameters *on this window* is the
  number in the table above, not an opinion -- and "capacity-matched control" is doing
  the real work in that sentence, not "gate", "highway", or "sparse_moe" by name.
- **"Does it forecast better" and "is it cheap to scale" are different questions, and
  this notebook answers both for MoE.** §5.4 trains dense and sparse routing side by
  side and reports the accuracy verdict directly, rather than only asserting the
  production argument. §5.5 measures the separate claim -- that sparse routing keeps
  compute flat as capacity grows -- which holds regardless of that verdict.
- **A negative result can still have a checkable cause.** §5.4 didn't stop at "16x
  more experts didn't help" -- it checked the router's actual usage and found only
  15 of 64 experts ever got chosen. "It didn't work" and "it didn't work, and here is
  why" are different levels of finished.

## Exercises

**E1 -- predict, then measure.** Before re-running this notebook, settle on a ranking
you expect for all seven trained models, and say whether you expect this series'
structure to change the §5.2 verdict from a null result to a real one. Check both.

**E2 -- turn the gate off.** Force `Gate`'s output to equal its input (patch the
forward pass, or initialise its linear layer to a large positive bias so the sigmoid
saturates near 1) and confirm the block's score converges toward `deeper`'s -- the
collapse argument from `1_pytorch_intro/02_nonlinearity` §2.1, for a gate instead of a
stacked linear layer.

**E3 -- more experts, dense vs sparse.** Re-run `make_model("moe")` with 2, 4, 8 and 16
experts (dense), each checked across the same epoch counts as §5.2, and compare against
`sparse_moe (N=64)`'s row already in §5.4's table. Does the dense router find anything
to split on as N grows, or does the score just track the parameter count -- and does
that answer differ from the routed version?

**E4 -- a different window.** Swap in `pedestrian_counts` (loaded the same way
`2_kernels_state_and_gates/optional_decomposition_and_nbeats` and `04_search` both load it) and re-run the whole
notebook. Does the ranking change on a series with an even sharper daily cycle?

**E5 -- k>1.** Re-run §5.5's timing sweep with `k=2` and `k=4` instead of `k=1`.
Predict first: does the flat region shift, and in which direction -- does it take more
experts, or fewer, before adding another one costs nothing extra? Then re-run §5.4's
`sparse_moe (N=64)` row at `k=2` and `k=4` -- does blending more routed experts change
the accuracy verdict, not just the timing one?

**E6 -- fix the collapse.** §5.4 found `sparse_moe (N=64)`'s router only ever used 15
of 64 experts on the validation set (checked at seed=0). Add an auxiliary
load-balancing loss to the training loop -- e.g. penalise the squared coefficient of
variation of per-expert routing counts within a batch, as Shazeer 2017 / the Switch
Transformer do -- and re-check: does a more balanced router change the accuracy
verdict, or only the balance?